[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509Files/blob/main/OPIM5509_Module3_Files/notebooks/Assignment4_ConvNets.ipynb)

# Assignment 4: Implementation of ConvNets (start early on this one!)
-------------------------------------
**Dr. Dave Wanik - OPIM 5509: Introduction to Deep Learning - University of Connecticut**

This is a practical ConvNet build, not a math problem: you will gather your own two-class image dataset, organize it the way Keras wants it, fit a convolutional neural network, and evaluate it properly - exactly what we did for cats vs. dogs in M3.2. *If you hit code problems, post on the Discussion Board so your classmates and I can help.*

**The one hard rule:** when you submit, I must be able to **Run all** on your notebook, top to bottom, on a fresh Colab runtime **without mounting Google Drive**. Everything - data download, folder prep, training, evaluation - has to happen inside the notebook.

## Your data (at least 200 images, two classes)

Pick any two classes you'd like to tell apart (mask vs. no mask, hot dog vs. not hot dog, your two favorite dog breeds, defective vs. good parts from a photo of your desk - be creative). Get **at least 100 images per class**. Two sanctioned ways to do it - use either, or both:

- **Route A - download them:** the `bing-image-downloader` demo below still works most days, but Bing is a scraper target and rate-limits. If you get fewer than 100, run several *different* search queries for the same class and combine them. (`icrawler` is a drop-in alternative if Bing stalls.)
- **Route B - bring your own:** zip your images into `myclass1/` and `myclass2/` folders, put the zip somewhere the notebook can `wget` it without a login (a **public** GitHub repo works great), and download + unzip it in a cell. This route is 100% reproducible and I'd honestly nudge you toward it.

Either way, the images land on the local runtime (folder icon on the left). Then use the **split helper** below to build `train/`, `validation/`, and `test/` partitions, each with one lowercase subfolder per class - **the folder names become your labels**, exactly like `cats/` and `dogs/`.

## Your model

Review `ConvNets_on_Small_Datasets_Cats_vs_Dogs.ipynb` and reuse its patterns: an `ImageDataGenerator` with `rescale=1/255`, three `flow_from_directory` generators (test with `shuffle=False`), a vanilla ConvNet with a sigmoid output, **early stopping on validation loss**, save the model as `.keras`, plot the loss and accuracy curves, then **evaluate every partition** and finish with a **classification report and confusion matrix with real class names** on the test partition. **You must build at least one vanilla ConvNet and evaluate it this way.** Above and beyond: data augmentation, transfer learning / fine-tuning with VGG16 (M3.3), or a Grad-CAM look at what your model is actually attending to (M3.2 interpretability notebook).

Your notebook should be laid out beautifully - headers, text cells, comments - at a quality you'd be proud to show a potential employer as evidence of your computer-vision skills.

**Rubric (100 points):**
* **(10)** A five-to-ten-sentence description of the problem you are solving with image classification (binary only for this assignment).
* **(40)** Data pipeline: images are downloaded or fetched inside the notebook (no Drive mount) and split into `train` / `validation` / `test` folders with one subfolder per class.
* **(20)** Fit a ConvNet - generators, early stopping, saved `.keras` model, loss and accuracy curves.
* **(20)** Evaluate it - scores for every partition, plus a classification report and confusion matrix (with class names) on the test partition.
* **(10)** At least five meaningful, thoughtful bullets on what you learned.

Severe deductions if the notebook lacks headers, text cells, and comments. **Zero points if the code is not "Run all" in its entirety before submitting** (I need to see your output).

Ask friends for help, but do your own work. I hope you enjoy this one.

## Route A - download images (Bing demo)

Run until you have enough; vary the query if Bing caps you. Delete this section if you use Route B.

In [ ]:
!pip -q install bing-image-downloader

In [ ]:
# if you don't get 100+, run it again with a DIFFERENT query for the same class (e.g. "elephant close up", "elephant herd")
from bing_image_downloader import downloader
downloader.download("elephants in africa",   limit=5, output_dir="images", adult_filter_off=True, force_replace=False)
downloader.download("kangaroo in australia", limit=5, output_dir="images", adult_filter_off=True, force_replace=False)

In [ ]:
!ls -R images | head -40

## Route B - bring your own zip (recommended for reproducibility)

Upload `my_images.zip` (containing `myclass1/` and `myclass2/`) to a **public** GitHub repo, then fetch it here. Delete this section if you use Route A.

In [ ]:
# example - replace with your own raw GitHub URL
# !wget -q -O my_images.zip https://raw.githubusercontent.com/<you>/<repo>/main/my_images.zip
# !unzip -q -o my_images.zip -d images

## Split into train / validation / test

One helper that turns `images/<class>/...` into the folder layout Keras reads labels from. Class folder names are lowercased so `cats`/`dogs`-style labels come out clean.

In [ ]:
import os, glob, random, shutil

def split_dataset(src="images", dst="data", train=0.7, val=0.15, seed=5509):
    """images/<class>/* -> data/{train,validation,test}/<class>/* (lowercase class names)."""
    random.seed(seed)
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")
    for cls in sorted(os.listdir(src)):
        files = [f for f in glob.glob(os.path.join(src, cls, "*")) if f.lower().endswith(exts)]
        random.shuffle(files)
        n_tr, n_va = int(len(files)*train), int(len(files)*val)
        parts = {"train": files[:n_tr], "validation": files[n_tr:n_tr+n_va], "test": files[n_tr+n_va:]}
        name = cls.lower().replace(" ", "_")
        for part, fs in parts.items():
            out = os.path.join(dst, part, name); os.makedirs(out, exist_ok=True)
            for f in fs: shutil.copy2(f, out)
        print(f"{name:20s} total={len(files):4d}  train={n_tr}  validation={n_va}  test={len(files)-n_tr-n_va}")

split_dataset()

train_dir      = "data/train"
validation_dir = "data/validation"
test_dir       = "data/test"

In [ ]:
# look at one image per class (are these really what you think they are?)
import matplotlib.pyplot as plt
from PIL import Image
classes = sorted(os.listdir(train_dir))
plt.figure(figsize=(6, 3))
for i, c in enumerate(classes):
    f = glob.glob(os.path.join(train_dir, c, "*"))[0]
    plt.subplot(1, len(classes), i+1); plt.imshow(Image.open(f).convert("RGB")); plt.title(c); plt.axis("off")
plt.show()

## Generators

Same recipe as cats vs. dogs: rescale by 1/255, fixed 150x150 target, `class_mode='binary'`, and **`shuffle=False` on the test generator** so predictions line up with labels.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rescale=1./255)          # (add augmentation args here for the train generator if you go above and beyond)
batch_size = 20

train_generator      = datagen.flow_from_directory(train_dir,      target_size=(150,150), batch_size=batch_size, class_mode="binary")
validation_generator = datagen.flow_from_directory(validation_dir, target_size=(150,150), batch_size=batch_size, class_mode="binary")
test_generator       = datagen.flow_from_directory(test_dir,       target_size=(150,150), batch_size=batch_size, class_mode="binary", shuffle=False)

print(train_generator.class_indices)   # <- your labels; alphabetical, first class = 0

## Build, fit, evaluate

Your turn. Build a vanilla ConvNet (Conv2D / MaxPooling2D stacks -> Flatten -> Dense -> sigmoid), compile with binary crossentropy, soft-code `steps_per_epoch`, train with early stopping, save the `.keras` model, plot the curves, evaluate every partition, and finish with the classification report + confusion matrix using `list(train_generator.class_indices)` as the target names. Everything you need is in `ConvNets_on_Small_Datasets_Cats_vs_Dogs.ipynb`.

In [ ]:
# your model here


In [ ]:
# fit with early stopping (soft-coded steps), save as .keras, plot loss + accuracy


In [ ]:
# evaluate every partition, then classification report + confusion matrix on the TEST partition with class names


## What I learned

At least five meaningful, thoughtful bullets.